# 05 — LTV Modeling

Trains the XGBoost LTV regressor (`src/ltv_model.py`) on `future_ltv_12m`,
the disclosed synthetic proxy target built in `feature_engineering.py`.

Per CLAUDE.md Section 6, `Churn` (ground truth) is excluded from the
feature set — only the churn classifier's **predicted probability** is
allowed in, never the true label. This notebook generates that probability
via **out-of-fold** predictions from the churn model (5-fold CV), so no
row's churn-model training peeks at its own churn-probability feature
before it's used here — a subtler leakage path than the ground-truth-label
exclusion alone would catch.

Prerequisite: run 01–04 first, or rebuild the pipeline inline below.

## Setup

In [1]:
import sys
sys.path.insert(0, "..")
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

from src.data_loader import load_raw_data
from src.preprocessing import clean_data
from src.feature_engineering import engineer_features
from src.segmentation import segment_customers
from src.churn_model import (
    prepare_features as churn_prepare_features,
    build_model as churn_build_model,
    compute_scale_pos_weight,
)
from src.ltv_model import (
    add_churn_probability_feature,
    get_feature_columns,
    cross_validate_ltv_model,
    train_final_model,
    save_model,
    predict_ltv,
    tier_predictions,
)

## Rebuild the pipeline input (through segmentation, k=5)

In [2]:
df = load_raw_data()
df = clean_data(df)
df = engineer_features(df, random_state=42)
df, cluster_profiles = segment_customers(df, k=5, random_state=42)

print("Pipeline input shape:", df.shape)
print("future_ltv_12m mean/median:", df["future_ltv_12m"].mean().round(2),
      "/", df["future_ltv_12m"].median().round(2))

Synthetic feature sanity check (mean by Churn):
       days_since_last_activity  days_active_last_90d  \
Churn                                                   
No                        10.63                 38.51   
Yes                       15.57                 22.12   

       avg_session_duration_min  weekend_activity_ratio  
Churn                                                    
No                        14.49                    0.28  
Yes                       10.13                    0.30  


Pipeline input shape: (7043, 37)
future_ltv_12m mean/median: 549.65 / 446.95


## Out-of-fold churn probability

5-fold Stratified CV on the churn model, predicting each fold's held-out
rows with a model that never trained on them. This is the
`churn_probability` feature `ltv_model.py` is allowed to use — the
ground-truth `Churn` label itself stays excluded.

In [3]:
X_churn, y_churn = churn_prepare_features(df)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_churn_proba = np.zeros(len(df))
for train_idx, val_idx in skf.split(X_churn, y_churn):
    spw = compute_scale_pos_weight(y_churn.iloc[train_idx])
    fold_model = churn_build_model(scale_pos_weight=spw, random_state=42)
    fold_model.fit(X_churn.iloc[train_idx], y_churn.iloc[train_idx])
    oof_churn_proba[val_idx] = fold_model.predict_proba(X_churn.iloc[val_idx])[:, 1]

pd.Series(oof_churn_proba).describe().round(3)

Prepared feature matrix: 7043 rows, 32 features (18 categorical), positive class rate=0.265


count    7043.000
mean        0.367
std         0.314
min         0.000
25%         0.056
50%         0.307
75%         0.652
max         0.983
dtype: float64

In [4]:
df = add_churn_probability_feature(df, oof_churn_proba)

Attached churn_probability feature (mean=0.367)


## Feature set

In [5]:
feature_cols = get_feature_columns(df)
print(f"{len(feature_cols)} features:")
feature_cols

33 features:


['Cluster_ID',
 'Contract',
 'Dependents',
 'DeviceProtection',
 'F_score',
 'InternetService',
 'M_score',
 'MonthlyCharges',
 'MultipleLines',
 'OnlineBackup',
 'OnlineSecurity',
 'PaperlessBilling',
 'Partner',
 'PaymentMethod',
 'PhoneService',
 'RFM_Score',
 'RFM_Segment',
 'R_score',
 'Segment_Name',
 'SeniorCitizen',
 'StreamingMovies',
 'StreamingTV',
 'TechSupport',
 'avg_addon_spend',
 'avg_session_duration_min',
 'churn_probability',
 'days_active_last_90d',
 'days_since_last_activity',
 'gender',
 'monthly_spend_trend_pct',
 'service_usage_interval_days',
 'tenure',
 'weekend_activity_ratio']

## Cross-validation

Plain K-Fold (no stratification needed — continuous target, not a
class-imbalance problem). Reports MAE, RMSE, R² per CLAUDE.md Section 6
(MAPE included as the optional extra).

In [6]:
cv_results = cross_validate_ltv_model(df, n_splits=5, random_state=42)
cv_results

Prepared LTV feature matrix: 7043 rows, 33 features (18 categorical), target mean=549.65


Evaluation: MAE=172.36, RMSE=246.23, R2=0.589, MAPE=84.1%


Fold 1/5 complete


Evaluation: MAE=177.19, RMSE=252.18, R2=0.568, MAPE=87.2%


Fold 2/5 complete


...

Fold 4/5 complete


Evaluation: MAE=175.85, RMSE=254.58, R2=0.576, MAPE=79.3%


Fold 5/5 complete


CV complete (5 folds). Mean MAE=175.80, RMSE=251.19, R2=0.574, MAPE=83.3%


,fold,mae,rmse,r2,mape
0,1,172.356102,246.233485,0.588579,84.061879
1,2,177.188727,252.178117,0.568183,87.242719
2,3,180.610907,256.313266,0.553692,83.907502
3,4,172.980948,246.623492,0.584285,81.760400
4,5,175.850072,254.581921,0.576274,79.340096


In [7]:
print("Mean CV metrics:")
cv_results[["mae", "rmse", "r2", "mape"]].mean().round(3)

Mean CV metrics:


mae     175.797
rmse    251.186
r2        0.574
mape     83.263
dtype: float64

## Final model (train/test split)

In [8]:
model, X_train, X_test, y_train, y_test, test_metrics = train_final_model(
    df, test_size=0.2, random_state=42
)
test_metrics

Prepared LTV feature matrix: 7043 rows, 33 features (18 categorical), target mean=549.65


Evaluation: MAE=171.00, RMSE=246.24, R2=0.589, MAPE=83.9%


{'mae': 170.9976326379715,
 'rmse': 246.24010946603315,
 'r2': 0.5885569069067785,
 'mape': 83.8779992531356}

## Leakage sanity check

`R²` should be well below the ~0.99 a naive `tenure x MonthlyCharges`
reconstruction would produce (CLAUDE.md Section 2's explicit warning).
A moderate R² here is a *good* sign — it means the model is learning from
RFM/segment/behavioral drivers, not just re-deriving the formula
`future_ltv_12m` was built from.

In [9]:
print(f"Test R² = {test_metrics['r2']:.3f}  (naive leakage reconstruction would be ~0.99)")

Test R² = 0.589  (naive leakage reconstruction would be ~0.99)


## Feature importance

In [10]:
importances = model.feature_importances_
feat_names = X_train.columns.tolist()
importance_df = (
    pd.DataFrame({"feature": feat_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
importance_df

,feature,importance
0,RFM_Segment,0.491449
1,Cluster_ID,0.167667
2,RFM_Score,0.049771
3,Contract,0.028472
4,MonthlyCharges,0.024222
5,tenure,0.017425
6,M_score,0.016592
7,OnlineSecurity,0.015944
8,TechSupport,0.015561
9,days_active_last_90d,0.011886


## Save the model

Writes to `models/ltv_model.joblib`.

In [11]:
save_model(model)

Saved LTV model to /home/claude/project/models/ltv_model.joblib


PosixPath('/home/claude/project/models/ltv_model.joblib')

## Tier the full dataset's predictions

Reuses `assign_ltv_tier()` from `feature_engineering.py` (via
`tier_predictions()`), applied to the model's actual predictions — this is
what CLAUDE.md Section 6 means by "predicted LTV values," distinct from
the synthetic-target tiering (`future_ltv_12m_tier`) done earlier for
EDA only.

In [12]:
from src.ltv_model import prepare_features

X_full, _ = prepare_features(df)
df["predicted_ltv"] = predict_ltv(model, X_full)
df["predicted_ltv_tier"] = tier_predictions(df["predicted_ltv"].to_numpy(), index=df.index)

print("Tier distribution (should be ~50/25/25 Low/Medium/High):")
df["predicted_ltv_tier"].value_counts(normalize=True).round(3)

Prepared LTV feature matrix: 7043 rows, 33 features (18 categorical), target mean=549.65


Tier distribution (should be ~50/25/25 Low/Medium/High):


predicted_ltv_tier
Low       0.50
Medium    0.25
High      0.25
Name: proportion, dtype: float64